# Homework 09: Feature Engineering

Three engineered features on a synthetic lending dataset (income, monthly spend, credit score,
region, and a default flag as the target), one of them a categorical encoding, each with a
rationale and a correlation check against default_flag.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.features import add_spend_income_ratio, add_credit_score_band, add_region_onehot

np.random.seed(0)
n = 100
df = pd.DataFrame({
    'income': np.random.normal(60000, 15000, n).astype(int),
    'monthly_spend': np.random.normal(2000, 600, n).astype(int),
    'credit_score': np.random.normal(680, 50, n).astype(int),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'default_flag': np.random.choice([0, 1], n, p=[0.8, 0.2]),
})
df.head()

,income,monthly_spend,credit_score,region,default_flag
0,86460,3129,661,East,0
1,66002,1191,668,North,0
2,74681,1237,734,West,0
3,93613,2581,712,South,1
4,88013,1296,712,East,1


## Feature 1: `spend_income_ratio`

`df['monthly_spend'] / df['income']`

In [2]:
df = add_spend_income_ratio(df)
df[['income', 'monthly_spend', 'spend_income_ratio']].head()

,income,monthly_spend,spend_income_ratio
0,86460,3129,0.036190
1,66002,1191,0.018045
2,74681,1237,0.016564
3,93613,2581,0.027571
4,88013,1296,0.014725


### Rationale for Feature 1

Income and spend on their own don't say much about financial strain. Someone spending 2,000 a
month on a 40,000 income is in a very different position than someone spending the same 2,000
on 120,000, even though `monthly_spend` looks identical either way. The ratio is the kind of
proportional relationship stage 08's EDA process exists to surface: two raw columns that only
mean something in relation to each other, not read separately.

### Correlation Check

In [3]:
corr_1 = df['spend_income_ratio'].corr(df['default_flag'])
print(f'Correlation with default_flag: {corr_1:.3f}')

fig, ax = plt.subplots(figsize=(5, 3.5))
df.boxplot(column='spend_income_ratio', by='default_flag', ax=ax)
ax.set_title('spend_income_ratio by default_flag')
plt.suptitle('')
fig.tight_layout()
fig.savefig('spend_income_ratio_by_default.png', dpi=110)
plt.close(fig)
print('Saved spend_income_ratio_by_default.png')

Correlation with default_flag: -0.036
Saved spend_income_ratio_by_default.png


## Feature 2: `credit_score_band`

`pd.cut(df['credit_score'], bins=[..., 579, 669, 739, 799, ...], labels=['Poor', ..., 'Exceptional'])`

In [4]:
df = add_credit_score_band(df)
df[['credit_score', 'credit_score_band']].head()

,credit_score,credit_score_band
0,661,Fair
1,668,Fair
2,734,Good
3,712,Good
4,712,Good


### Rationale for Feature 2

Lending risk doesn't move smoothly with credit score, it moves in bands: a 580 and a 620 are
both "Poor" in practice and get treated about the same by an underwriter, while the jump from
619 to 620 (Poor to Fair) matters a lot more than the raw number difference suggests. A model
given the raw score has to learn that non-linearity itself; handing it the band directly encodes
domain knowledge instead of hoping the model finds the same thresholds from data alone.

### Correlation Check

In [5]:
default_rate_by_band = df.groupby('credit_score_band', observed=True)['default_flag'].mean()
print('Default rate by credit_score_band:')
print(default_rate_by_band)

fig, ax = plt.subplots(figsize=(5, 3.5))
default_rate_by_band.plot(kind='bar', ax=ax, color='#2c5cc5')
ax.set_ylabel('default rate')
ax.set_title('Default rate by credit_score_band')
fig.tight_layout()
fig.savefig('default_rate_by_band.png', dpi=110)
plt.close(fig)
print('Saved default_rate_by_band.png')

Default rate by credit_score_band:
credit_score_band
Poor         0.000000
Fair         0.142857
Good         0.297872
Very Good    0.250000
Name: default_flag, dtype: float64
Saved default_rate_by_band.png


## Feature 3: `region` one-hot encoding (categorical)

`pd.get_dummies(df, columns=['region'], prefix='region')`

The lecture shows three options: one-hot, label, and frequency encoding. Label encoding is the
wrong choice here since it would assign region an arbitrary numeric order (North=0, South=1, ...)
that a model could easily mistake for a real ranking, when region has no natural order. Frequency
encoding keeps things compact but throws information away: if two different regions happen to
have a similar share of rows, frequency encoding gives them nearly the same value even though
they're not the same region. One-hot avoids both problems, at the cost of one extra column per
category, which is a fine trade with only 4 regions.

In [6]:
df_encoded = add_region_onehot(df)
region_cols = [c for c in df_encoded.columns if c.startswith('region_')]
df_encoded[region_cols].head()

,region_East,region_North,region_South,region_West
0,True,False,False,False
1,False,True,False,False
2,False,False,False,True
3,False,False,True,False
4,True,False,False,False


### Correlation Check

In [7]:
region_corr = df_encoded[region_cols + ['default_flag']].corr()['default_flag'].drop('default_flag')
print('Correlation of each region dummy with default_flag:')
print(region_corr)

fig, ax = plt.subplots(figsize=(5, 3.5))
region_corr.plot(kind='bar', ax=ax, color='#2c5cc5')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('correlation with default_flag')
ax.set_title('Region (one-hot) vs default_flag')
fig.tight_layout()
fig.savefig('region_onehot_corr.png', dpi=110)
plt.close(fig)
print('Saved region_onehot_corr.png')

Correlation of each region dummy with default_flag:
region_East    -0.116131
region_North    0.009324
region_South    0.152691
region_West    -0.048951
Name: default_flag, dtype: float64
Saved region_onehot_corr.png


None of the three correlations above are large, which is expected since this dataset was
generated with `default_flag` drawn independently of every other column, there's no real signal
for any feature to find. That's fine for this exercise: the point is that the features are
implemented correctly and checked properly, not that they find a strong effect in data that
was never given one. On the real project dataset, the same three checks would be read for
actual signal, not just correctness.